<a href="https://colab.research.google.com/github/justorfc/Est_Python_R_2026_1/blob/main/8_Semana_08_Sesion_02_Aplicacion_Bondad_Ajuste.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Vamos a diseñar la segunda sesión de esta semana.

En esta parte práctica, es fundamental que veamos qué ocurre cuando intentamos forzar un modelo matemático equivocado sobre los datos observados.

Esto nos enseñará a no confiar ciegamente en la intuición visual.

### 📘 Estructura del Notebook: Semana 8 - Sesión 2

**Título del Notebook:** `Semana_08_Sesion_02_Aplicacion_Bondad_Ajuste.ipynb`

#### **Celda 1: Markdown (Encabezado y Fase 1: IA)**

# SEMANA 8: Pruebas de Bondad de Ajuste
**Asignatura:** Estadística Aplicada con Python y R
**Sesión 2:** Aplicación Práctica, Rechazo de Modelos y Chi-cuadrado.

## Fase 1: Actividad "Estudia y Aprende" (Aplicación)
Antes de ejecutar las pruebas estadísticas, pídele a tu IA que te muestre la sintaxis de las funciones en Python y R, y que te explique cómo interpretar un rechazo del modelo.

**PROMPT 2 - APLICACIÓN:**
> Actúa como tutor experto en pruebas de bondad de ajuste con Python y R.
> 1. Muéstrame cómo aplicar KS a una distribución Normal.
> 2. Cómo aplicar Anderson-Darling.
> 3. Cómo aplicar Chi-cuadrado para datos discretos.
> 4. Explica cómo interpretar los resultados.
> 5. Explica qué significa rechazar o no rechazar el modelo.
> No solo muestres código; interpreta los resultados como lo haría un ingeniero.

#### **Celda 2: Markdown (Fase 2: Demostración Docente - Contexto)**

## Fase 2: Demostración Docente en Python
Ayer vimos un caso donde el modelo "pasaba" la prueba. Hoy haremos lo contrario: intentaremos ajustar una distribución **Normal** a datos que en la realidad son fuertemente asimétricos (como los caudales extremos de un río o la resistencia a la fatiga). Veremos cómo las pruebas KS y AD detectan este error.

Además, aplicaremos la prueba **Chi-cuadrado**, que es la herramienta predilecta para evaluar variables **discretas** (conteos, como el número de grietas o fallas por día).

**Pregunta clave:** ¿Es razonable usar este modelo para decisiones de ingeniería?

#### **Celda 3: Código Python (Rechazando el modelo equivocado con KS y AD)**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
np.random.seed(123)

# 1. GENERAMOS DATOS ASIMÉTRICOS (Lognormal)
# Contexto: Concentración de un contaminante en el suelo (Ing. Agroindustrial/Agrícola)
datos_reales = stats.lognorm.rvs(s=0.8, scale=50, size=150)

# Intentamos forzar un ajuste NORMAL (Incorrecto)
media_est = np.mean(datos_reales)
std_est = np.std(datos_reales)
datos_estandarizados = (datos_reales - media_est) / std_est

print("--- EVALUANDO AJUSTE NORMAL SOBRE DATOS ASIMÉTRICOS ---")

# Prueba KS
ks_stat, ks_pvalor = stats.kstest(datos_estandarizados, 'norm')
print(f"KS Valor-p: {ks_pvalor:.5f}")

# Prueba AD
ad_result = stats.anderson(datos_reales, dist='norm')
print(f"AD Estadístico: {ad_result.statistic:.4f}")
print(f"AD Valor Crítico (5%): {ad_result.critical_values[2]:.4f}")

# Interpretación Ingenieril:
# - El Valor-p de KS es diminuto (mucho menor a 0.05). RECHAZAMOS la Hipótesis Nula (H0).
# - El Estadístico de AD es inmensamente mayor que el valor crítico. RECHAZAMOS H0.
# Conclusión: Asumir que estos datos son Normales es un error estadístico grave.
# Si diseñamos una planta de tratamiento basándonos en una distribución Normal para estos datos,
# la planta colapsará ante las concentraciones extremas reales (la cola derecha).

#### **Celda 4: Código Python (Prueba Chi-cuadrado para datos discretos)**

In [ ]:
# 2. PRUEBA CHI-CUADRADO (Datos Discretos)
# Contexto: Número de fallas mecánicas por semana en una flota de tractores.
# Queremos saber si este conteo sigue una distribución de Poisson.

# Frecuencias Observadas (Días que tuvieron 0, 1, 2, o 3+ fallas)
# En 100 semanas evaluadas:
frec_observadas = np.array([45, 35, 15, 5])

# Frecuencias Esperadas (Calculadas teóricamente bajo un modelo de Poisson con lambda = 0.8)
frec_esperadas = np.array([44.9, 35.9, 14.4, 4.8])

# Ajuste menor por redondeo para que ambas sumen exactamente lo mismo (requisito de la prueba)
frec_esperadas = frec_esperadas * (frec_observadas.sum() / frec_esperadas.sum())

print("\n--- PRUEBA CHI-CUADRADO (Ajuste a Poisson) ---")
chi2_stat, chi2_pvalor = stats.chisquare(f_obs=frec_observadas, f_exp=frec_esperadas)

print(f"Estadístico Chi-cuadrado: {chi2_stat:.4f}")
print(f"Valor-p: {chi2_pvalor:.4f}")

# Interpretación:
# El Valor-p es muy alto (aprox. 0.99), es decir, muy superior a 0.05.
# NO hay evidencia para rechazar H0.
# Conclusión: Podemos modelar las fallas de los tractores usando la distribución de Poisson
# para planificar nuestros mantenimientos y compra de repuestos.

#### **Celda 5: Markdown (Transición a R y RMarkdown)**

## Trabajo Autónomo: Transición a R y RMarkdown

R tiene una función nativa muy robusta para la prueba Chi-cuadrado.

**Paso 1: Validación en Colab (Entorno R)**
1. Abre un nuevo Notebook en Colab, asegúrate de cambiar el entorno a **R**.
2. Ejecuta el siguiente código para verificar el ajuste discreto.

```R
# Prueba Chi-cuadrado en R
# Vectores de frecuencias
observados <- c(45, 35, 15, 5)
esperados_prob <- c(0.449, 0.359, 0.144, 0.048) # Probabilidades teóricas

print("--- Prueba Chi-cuadrado ---")
# chisq.test compara conteos observados contra probabilidades esperadas (p)
chi_test <- chisq.test(x = observados, p = esperados_prob)
print(chi_test)

# Interpretación: Revisa el p-value impreso en la consola.
# Si es mayor a 0.05, el modelo discreto propuesto es un buen ajuste.

```

**Paso 2: Documentación en Posit Cloud y Despliegue en RPubs**

1. En **Posit Cloud**, crea tu documento **RMarkdown**.
2. Transfiere el código R.
3. Redacta la conclusión final: *Explica detalladamente por qué es un peligro en la ingeniería asumir una distribución probabilística solo viendo el histograma, y por qué el Valor-p nos salva de cometer errores de sobrediseño o subdiseño.*
4. Compila a HTML y publica en **RPubs**.

#### **Celda 6: Markdown (CIERRE DEL TEMA Y HOJA EVALUABLE)**

## CIERRE DEL TEMA: Generación de Resumen Guía

Hemos finalizado este bloque crucial. Ahora sabes cómo defender estadísticamente la elección de un modelo. Prepara tu síntesis usando el siguiente prompt.

**PROMPT DE CIERRE GLOBAL:**
> Genera un resumen estructurado para escribir a mano en UNA sola hoja.
> Formato obligatorio:
> A) Idea central (1-2 líneas).
> B) 6-10 viñetas organizadas lógicamente.
> C) 3 relaciones clave (por qué/cómo).
> D) 1 ejemplo aplicado a ingeniería.
> E) 3 preguntas de autoevaluación con respuesta.
> F) Cierre: "Hoy aprendí que ..."

⚠️ **RECUERDA (EVIDENCIA EVALUABLE):**
En el reverso de tu hoja física titulada *"Pruebas de Bondad de Ajuste en Ingeniería"*, escribe a mano:
* La idea central sobre evaluar modelos vs. realidad.
* Qué es la bondad de ajuste.
* Las Hipótesis: $H_0$ y $H_1$.
* Interpretación estricta del valor-p (¿Cuándo rechazamos el modelo?).
* Las diferencias clave entre Kolmogorov-Smirnov, Anderson-Darling y Chi-cuadrado.
* Un ejemplo aplicado.
* Las 3 preguntas con respuestas.
* Tu reflexión final ("Hoy aprendí que...").

Con la Semana 8 concluida, ya se tienen las bases probabilísticas sólidas y validadas matemáticamente para adentrarse de lleno en los modelos predictivos.